In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
import os
from copy import deepcopy
import time
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import pandas as pd
from core.ts_scaling import get_ts_scaling_names
from sm.surrogates_feasible_space import (Feasibility_LP, get_result_column_names, save_unique_vertex_points)
from sm.feasible_space_expl import (sample_min_max, sample_most_critical_and_feasible,
                                     sample_random, running_unique_counts, Convex_Hull_Explorer)
from core.helpers import (get_config_file, get_data_out, get_data_computed_in, get_process_names,
                         set_rwth_color)

PROJECT_FILE = 'project_ID13_5.yaml'
CASE = 'JP1'
ROOT = os.path.dirname(os.path.dirname(os.getcwd()))
DATA_IN = os.path.join(ROOT, 'data', CASE, 'Input_Data')
CONFIG = get_config_file(DATA_IN, PROJECT_FILE)

DATA_COMPUTED_IN = get_data_computed_in(DATA_IN, PROJECT_FILE)
DATA_OUT         = get_data_out(DATA_IN, PROJECT_FILE)

### A: convex_hull_sample_generator:

In [ ]:
ts_scaling_names = get_ts_scaling_names(DATA_IN, PROJECT_FILE, only_feasibility_affecting=True)
process_names = get_process_names(DATA_IN, PROJECT_FILE, only_feasibility_affecting=True)

N = 200_000

df_sample = pd.concat(
    [
        sample_min_max(ts_scaling_names, process_names),
        sample_most_critical_and_feasible(ts_scaling_names, process_names),
        sample_random(ts_scaling_names, process_names, N)
    ],
    axis=0, ignore_index=True)

print("Done building df_sample")
df_sample.to_csv(
    os.path.join(DATA_COMPUTED_IN, f'convex_hull_vertex_{len(df_sample)}.csv'),
    index=True, index_label='id')
df_sample

In [ ]:
df_sample = pd.read_csv(
    os.path.join(DATA_COMPUTED_IN, f'convex_hull_vertex_{len(df_sample)}.csv'), index_col='id'
)
df_sample.index.name = None

print(len(df_sample))

### B: convex_hull_explorer

In [ ]:
from sm.routines import explore_convex_hull

ch_explorer = Convex_Hull_Explorer(DATA_IN, PROJECT_FILE)

df_result = explore_convex_hull(
    df_in= df_sample,
    data_in_path=DATA_IN,
    project_file=PROJECT_FILE,
    debug=False
)

In [ ]:
# plot number unique vertex points over the number of iterations:
column_names_norm = ch_explorer.get_result_column_names(True)

df_sample = pd.read_csv(os.path.join(DATA_OUT, 'convex_hull_vertex_results.csv'), index_col='id')

# remove rows containing NaN values
df_sample = df_sample.dropna(axis=0, how='any')
print(len(df_sample))

df_sample.index.name = None

values = df_sample[column_names_norm].values
unique_rows = np.unique(values, axis=0)

count = running_unique_counts(values)

In [ ]:

plt.close('all')


fig, ax = plt.subplots()

ax.plot(range(len(count)), count, label='data')
old_ylim = ax.get_ylim()

ax.plot([0, len(count)],[0, len(count)], color='k', linestyle='--',label='winkelhalbierende')

ax.set_xlabel("exploration runs")
ax.set_ylabel("number of unique points")
ax.legend(loc='best')

ax.grid()
ax.set_ylim(old_ylim)
plt.tight_layout()
plt.show()

### C: Feasibility LP as Feasibility Predictor

In [ ]:
df_sample_L = pd.read_csv(os.path.join(DATA_OUT, 'convex_hull_vertex_results_6974.csv'))

df_unique_L = df_sample_L.drop_duplicates(
    subset=column_names_norm, keep='first'
)
len(df_unique_L)

In [ ]:
test_df = pd.read_csv(os.path.join(DATA_COMPUTED_IN, 'operational_opt_var_ts_lhs_sample_100000.csv'))

feasiblity_lp = Feasibility_LP(
    DATA_IN, PROJECT_FILE,
    df_vertices=df_unique_L
)

feasiblity_lp.vertices.shape

In [ ]:

(feasibility, dt) = feasiblity_lp.predict_feasibility(test_df)


In [ ]:
test_df['convex_hull_feasibility'] = feasibility
test_df['dt_feasibility_lp'] = dt

In [ ]:
test_df.to_csv(
    os.path.join(DATA_OUT, "feasiblity_lp_predictions.csv"), index=False
)

In [ ]:
# Check the number of False classifications

df_IST = pd.read_csv(os.path.join(DATA_OUT, 'op_var_max_overcapacity_100000.csv'))
df_IST.columns

In [ ]:
n_infeasible_IST = (df_IST['overcapacity'] < 0).sum()
print(f"n_infeasible_IST = {n_infeasible_IST}")
n_feasible_IST = (df_IST['overcapacity'] >= 0).sum()
print(f"n_feasible_IST = {n_feasible_IST}")


n_false_feasible = (
    (df_IST['overcapacity'] < 0).to_numpy() & test_df['convex_hull_feasibility'].to_numpy()
    ).sum()
print(f"\nn_false_feasible = {n_false_feasible}")

n_true_feasible = (
    (df_IST['overcapacity'] >= 0).to_numpy()  & test_df['convex_hull_feasibility'].to_numpy()
    ).sum()

print(f"n_true_feasible = {n_true_feasible}")

n_false_infeasible = (
    (df_IST['overcapacity'] >= 0).to_numpy() & ~test_df['convex_hull_feasibility'].to_numpy()
    ).sum()
print(f"n_false_infeasible = {n_false_infeasible}")

n_true_infeasible = (
    (df_IST['overcapacity'] < 0).to_numpy() & ~test_df['convex_hull_feasibility'].to_numpy()
    ).sum()
print(f"n_true_infeasible = {n_true_infeasible}")

### D: Check how many unique data points are required:

In [ ]:
# generate smaller sets of unique points
df_sample = pd.read_csv(os.path.join(DATA_OUT, 'convex_hull_vertex_results_300k.csv'), index_col='id')

# df_sample.columns
# Filter out doublicates:
column_names_norm = get_result_column_names(DATA_IN, PROJECT_FILE, True)

df_unique = df_sample.drop_duplicates(
    subset=column_names_norm, keep='first'
)

n_unique_list = [50, 100, 200, 500, 1000, 2000, 3000, 5000, len(df_unique)]
for n_unique in n_unique_list:
    df_unique[:n_unique].to_csv(os.path.join(DATA_OUT, f'convex_hull_vertex_results_{n_unique}.csv'), index=False)

# # Save full dataframe:

df_unique.to_csv(os.path.join(DATA_OUT, f'convex_hull_vertex_results_{len(df_unique)}.csv'), index=False)

In [ ]:
save_unique_vertex_points(
    DATA_IN, PROJECT_FILE,
    full_vertex_csv_name='convex_hull_vertex_results.csv',
    unique_vertex_name='convex_hull_vertex_results.csv',
    n_unique_list=[5000],
    add_all_unique_vertices=True
)

In [ ]:

n_unique_list = [50, 100, 200, 500, 1000, 2000, 3000, 5000, len(df_unique)]
for n_unique in n_unique_list:
    print(20*'#' + f"\n\t\t{n_unique}\n" + 20*'#')
    test_df = pd.read_csv(os.path.join(DATA_OUT, 'op_var_100000_test.csv'))

    df_unique_vertices = pd.read_csv(
        os.path.join(DATA_OUT, f'convex_hull_vertex_results_{n_unique}.csv'))

    feasiblity_lp = Feasibility_LP(
        DATA_IN, PROJECT_FILE,
        df_vertices=df_unique_vertices
    )

    (feasibility, dt) = feasiblity_lp.predict_feasibility(test_df)

    # Write results:
    test_df['convex_hull_feasibility'] = feasibility
    test_df['dt_feasibility_lp'] = dt

    test_df.to_csv(
        os.path.join(DATA_OUT, f"feasiblity_lp_predictions_{n_unique}.csv"), index=False
    )


In [ ]:
# Check the number of False classifications


# generate smaller sets of unique points
df_sample = pd.read_csv(os.path.join(DATA_OUT, 'convex_hull_vertex_results.csv'), index_col='id')
# Filter out doublicates:
column_names_norm = get_result_column_names(DATA_IN, PROJECT_FILE, True)

df_unique = df_sample.drop_duplicates(
    subset=column_names_norm, keep='first'
)


TF_list = []
TI_list = []
FF_list = []
FI_list = []
mean_time_list = []

n_unique_list = [50, 100, 200, 500, 1000, 2000, 3000, 5000, len(df_unique)]
for n_unique in n_unique_list:

    print(f"\n####### {n_unique} #######")
    test_df = pd.read_csv(
        os.path.join(DATA_OUT, f"feasibility_lp_prediction_{n_unique}.csv")
    )
    FF = (
        (df_IST['overcapacity'] < 0).to_numpy() & test_df['convex_hull_feasibility'].to_numpy()
        ).sum()
    print(f"\nFF = {FF}")

    TF = (
        (df_IST['overcapacity'] >= 0).to_numpy()  & test_df['convex_hull_feasibility'].to_numpy()
        ).sum()

    print(f"TF = {TF}")

    FI = (
        (df_IST['overcapacity'] >= 0).to_numpy() & ~test_df['convex_hull_feasibility'].to_numpy()
        ).sum()
    print(f"FI = {FI}")

    TI = (
        (df_IST['overcapacity'] < 0).to_numpy() & ~test_df['convex_hull_feasibility'].to_numpy()
        ).sum()
    print(f"TI = {TI}")

    TF_list.append(TF)
    TI_list.append(TI)
    FF_list.append(FF)
    FI_list.append(FI)

    mean_time_list.append(test_df['dt_feasibility_lp'].mean())
TF = np.array(TF_list)
TI = np.array(TI_list)
FF = np.array(FF_list)
FI = np.array(FI_list)

In [ ]:
class Feas_LP_Evaluater():

    def __init__(self, data_in_path:str, project_file:str):

        self.n_days = get_config_file(DATA_IN, project_file)['time_series_aggregation']['noTypicalPeriods']
    
        self.data_in_path = data_in_path
        self.project_file = project_file
        self.data_out_path = get_data_out(data_in_path, project_file)
    
        self.FP = []
        self.TP = []
        self.FN = []
        self.TN = []
        self.mean_times = []
        self.unique_list = []
    
        self.FOR = [] # False Omission Rate
        self.FDR = [] # False Discovery Rate
        self.PPV = [] # Positive Predictive Value
        self.NPV = [] # Negative Predictive Value

    def define_classification_metrics(self, df_vertices_filename:str, n_unique_list:list,
                                      df_IST_filename:str='op_var_100000_test.csv'):

        # Load vertices used for testing
        df_vertex = pd.read_csv(
            os.path.join(self.data_out_path, df_vertices_filename), index_col='id')

        # Filter out doublicates:
        column_names_norm = get_result_column_names(self.data_in_path, self.project_file, True)
        df_unique = df_vertex.drop_duplicates(
            subset=column_names_norm, keep='first'
        )

        n_unique_list = deepcopy(n_unique_list)
        if len(df_unique) not in n_unique_list:
            n_unique_list.append(len(df_unique))

        self.unique_list = n_unique_list

        # Load IST data
        df_IST = pd.read_csv(os.path.join(self.data_out_path, df_IST_filename))
    
        for n_unique in n_unique_list:
            print(f"\n####### {n_unique} #######")
            test_df = pd.read_csv(
                os.path.join(self.data_out_path, f"feasibility_lp_prediction_{n_unique}.csv")
            )
            FF = (
                (df_IST['overcapacity'] < 0).to_numpy() & test_df['convex_hull_feasibility'].to_numpy()
                ).sum()
            # print(f"\nFF = {FF}")

            TF = (
                (df_IST['overcapacity'] >= 0).to_numpy()  & test_df['convex_hull_feasibility'].to_numpy()
                ).sum()

            # print(f"TF = {TF}")

            FI = (
                (df_IST['overcapacity'] >= 0).to_numpy() & ~test_df['convex_hull_feasibility'].to_numpy()
                ).sum()
            # print(f"FI = {FI}")

            TI = (
                (df_IST['overcapacity'] < 0).to_numpy() & ~test_df['convex_hull_feasibility'].to_numpy()
                ).sum()
            # print(f"TI = {TI}")

            self.TP.append(TF)
            self.TN.append(TI)
            self.FP.append(FF)
            self.FN.append(FI)

            self.mean_times.append(test_df['dt_feasibility_lp'].mean())
        
        # convert to np arrays
        self.TP = np.array(self.TP)
        self.TN = np.array(self.TN)
        self.FP = np.array(self.FP)
        self.FN = np.array(self.FN)

        # compute metrics
        self.FOR = self.FN / (self.TN + self.FN) * 100
        self.FDR = self.FP / (self.TP + self.FP) * 100
        self.PPV = self.TP / (self.TP + self.FP) * 100
        self.NPV = self.TN / (self.TN + self.FN) * 100

n_unique_list_base = [50, 100, 200, 500, 1000, 2000, 3000, 5000]

feas_lp_eval_5 = Feas_LP_Evaluater(DATA_IN, "project_ID13_5.yaml")
feas_lp_eval_5.define_classification_metrics('convex_hull_vertex_results.csv', n_unique_list_base)

feas_lp_eval_10 = Feas_LP_Evaluater(DATA_IN, "project_ID13_10.yaml")
feas_lp_eval_10.define_classification_metrics('convex_hull_vertex_results.csv', n_unique_list_base)

feas_lp_eval_50 = Feas_LP_Evaluater(DATA_IN, "project_ID13_50.yaml")
feas_lp_eval_50.define_classification_metrics('convex_hull_vertex_results.csv', n_unique_list_base)

feas_lp_eval_100 = Feas_LP_Evaluater(DATA_IN, "project_ID13_100.yaml")
feas_lp_eval_100.define_classification_metrics('convex_hull_vertex_results.csv', n_unique_list_base)


In [ ]:
# Negative Predictive Value
plt.close('all')
from matplotlib.lines import Line2D
TEXTWIDTH = 6.842519441
MARKERSIZE_S = 25
COLOR_NF1 = set_rwth_color('blue', 100)

plt.rcParams['text.usetex'] = True  
plt.rcParams['font.family'] = 'serif'

fig, axs = plt.subplots(2, 2, figsize=(0.8 * TEXTWIDTH, 4.2), constrained_layout=True)

feas_lps = [feas_lp_eval_5, feas_lp_eval_10, feas_lp_eval_50, feas_lp_eval_100]

for j, (ax, feas_lp) in enumerate(zip(axs.flatten(), feas_lps)):
    
    for i, n_unique in enumerate(feas_lp.unique_list):

        if feas_lp.FN[i] == 0:
            color = COLOR_NF1
            marker = '^'
        else:
            color = 'k'
            marker = 'v'
        
        ax.plot(feas_lp.NPV[i], feas_lp.mean_times[i] , marker=marker, markersize=np.sqrt(MARKERSIZE_S), color=color, zorder=3)
        offset = -2 #if n_unique != 50 else -8

        ax.text(feas_lp.NPV[i] + offset,
                feas_lp.mean_times[i],
                s=f"${n_unique}$",#\n{FI[i]}",
                verticalalignment='center',horizontalalignment='right')

    ax.set_yscale('log')
    ax.grid(which='both')
    ax.set_title(f"{feas_lp.n_days} days")
    ax.set_ylim(4/10000, 50/1000)
    ax.set_xlim(30, 102)
    # Legend:
    legend_elements = [
        Line2D([1], [1], marker='^', color='none',
            markerfacecolor=COLOR_NF1,
            markeredgecolor=COLOR_NF1, markersize=np.sqrt(MARKERSIZE_S),
            label=r'$\mathrm{FN} = 0$'),
        Line2D([1], [1], marker='v', color='none',
            markerfacecolor='k', markersize=np.sqrt(MARKERSIZE_S),
            label=r'$\mathrm{FN} > 0$'),
    ]

    ax.legend(handles=legend_elements, loc='best')

    # Only show labels and ticklabels on left and bottom plots:
    if j in [0, 2]:
        ax.set_ylabel("mean prediction time in s")
    else:
        ax.set_yticklabels([])
    if j in [2, 3]:
        ax.set_xlabel(r"NPV in \%")
    else:
        ax.set_xticklabels([])

    ax.xaxis.set_major_locator(MultipleLocator(20))
    ax.xaxis.grid(True, which='major')
    ax.xaxis.set_minor_locator(MultipleLocator(5))
    ax.xaxis.grid(True, which='minor', linestyle='-', linewidth=0.3)


data_out = get_data_out(DATA_IN, "project_ID13_100.yaml")
# Save plot:
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)

plt.savefig(
    os.path.join(data_out, "plots", "fig05_ch_classification_all_day_LOG_SI.pdf"), dpi=300, bbox_inches='tight')

In [ ]:
# plot only for 10 days:
# Negative Predictive Value
plt.close('all')
from matplotlib.lines import Line2D

MARKERSIZE_S = 25
COLOR_NF1 = set_rwth_color('blue', 100)

plt.rcParams['text.usetex'] = True  
plt.rcParams['font.family'] = 'serif'

fig, ax = plt.subplots(1, 1, figsize=(3.296726719, 2.4), constrained_layout=True)

feas_lp = feas_lp_eval_50


for i, n_unique in enumerate(feas_lp.unique_list):

    if feas_lp.FN[i] == 0:
        color = COLOR_NF1
        marker = '^'
    else:
        color = 'k'
        marker = 'v'
    
    ax.plot(feas_lp.NPV[i], feas_lp.mean_times[i] , marker=marker, markersize=np.sqrt(MARKERSIZE_S), color=color, zorder=3)
    offset = -2 #if n_unique != 50 else -8

    ax.text(feas_lp.NPV[i] + offset,
            feas_lp.mean_times[i],
            s=f"${n_unique}$",#\n{FI[i]}",
            verticalalignment='center',horizontalalignment='right')

ax.set_yscale('log')
ax.grid(which='both')
ax.set_xlabel(r"NPV in \%")
ax.set_ylabel("mean prediction time in s")
# ax.set_title(f"{feas_lp.n_days} days")
ax.set_ylim(4/10000, 50/1000)
ax.set_xlim(30, 102)
# Legend:
legend_elements = [
    Line2D([1], [1], marker='^', color='none',
        markerfacecolor=COLOR_NF1,
        markeredgecolor=COLOR_NF1, markersize=np.sqrt(MARKERSIZE_S),
        label=r'$\mathrm{FN} = 0$'),
    Line2D([1], [1], marker='v', color='none',
        markerfacecolor='k', markersize=np.sqrt(MARKERSIZE_S),
        label=r'$\mathrm{FN} > 0$'),
]

ax.xaxis.set_major_locator(MultipleLocator(20))
ax.xaxis.grid(True, which='major')
ax.xaxis.set_minor_locator(MultipleLocator(5))
ax.xaxis.grid(True, which='minor', linestyle='-', linewidth=0.3)

ax.legend(handles=legend_elements, loc='best')
# Save plot:
data_out = get_data_out(DATA_IN, "project_ID13_100.yaml")
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)

plt.savefig(
    os.path.join(data_out, "plots", f"fig05_ch_classification_{feas_lp.n_days}_LOG.pdf"), dpi=300, bbox_inches='tight')

# Try to use convex hull

In [ ]:
# df_sample_L = pd.read_csv(os.path.join(DATA_OUT, 'vertex_results_large.csv'), index_col='id')
df_sample_L = pd.read_csv(os.path.join(DATA_OUT, 'convex_hull_vertex_results_100.csv'), index_col='id')

column_names_norm = get_result_column_names(
    DATA_IN, PROJECT_FILE, normed_keys=True)

df_unique_L = df_sample_L.drop_duplicates(
    subset=column_names_norm, keep='first'
)

feasiblity_lp = Feasibility_LP(
    DATA_IN, PROJECT_FILE,
    min_max_import_path=os.path.join(DATA_OUT,"min_max_values.yaml"),
    df_vertices=df_unique_L
)
